# LLM QLoRA 微调训练 notebook

**llm-finetune | Unsloth加速 | 推理监控 | BERTScore+Rouge**

---

## 使用方法

**Step 1** `pip install -r requirements.txt`

**Step 2** 把数据文件放入 `./datasets/`

**Step 3** 修改 Cell 1 的模型路径和数据目录

**Step 4** 按顺序 Run 每个 Cell

---

## Stage速查

|Stage|Cell|作用|何时跑|
|---|---|---|---|
|0|Cell 0|环境检查|只需一次|
|1|Cell 1|配置参数|只需一次|
|2|Cell 2|GPU检测|只需一次|
|3|Cell 3|数据加载|数据不变可跳过|
|4|Cell 4|模型加载|模型不变可跳过|
|5|Cell 5|训练|每次训练|
|6|Cell 6|评估+推理|训练完成后|
|7|Cell 7|独立推理测试|随时可跑|
|8|Cell 8|导出模型|训练完成后|

---

## 任务类型说明

|task_type|说明|适用场景|
|---|---|---|
|instruction|标准指令微调|推荐，通用|
|cot|思维链|需含Complex_CoT字段|
|classification|文本分类|输入→类别|
|text_completion|文本续写|续写|


---

## Stage 1: 配置参数

**必须修改三项**：

|参数|说明|示例|
|---|---|---|
|name_or_path|模型路径|Qwen/Qwen3-8B|
|dataset_dir|数据目录|./datasets|
|task_type|任务类型|instruction|

**task_type**: instruction / cot / classification / text_completion

**常用参数**：
- num_epochs（默认3，数据少用3，数据多用1-2）
- lora_r（默认16，显存紧张降到8）
- max_seq_length（默认2048，显存紧张降到1024）


In [ ]:
import yaml

# 加载 config.yaml
config = yaml.safe_load(open('config.yaml', encoding='utf-8'))

# 【必须修改】以下三项根据你的环境填写
# ① 模型路径：HuggingFace ID（自动下载）或本地目录
config['model']['name_or_path'] = 'Qwen/Qwen3-8B'

# ② 数据集目录：放入 json/jsonl/csv/tsv/txt 文件
config['data']['dataset_dir'] = './datasets'

# ③ 任务类型：instruction / cot / classification / text_completion
config['direction']['task_type'] = 'instruction'

# 【可选修改】
config['training']['num_epochs'] = 3
config['qlora']['r'] = 16
config['training']['max_seq_length'] = 2048
config['unsloth']['enabled'] = True
# config['wandb']['enabled'] = True

# 打印配置确认
print('=' * 50)
print('当前配置确认')
print('=' * 50)
print('模型:', config['model']['name_or_path'])
print('数据集:', config['data']['dataset_dir'])
print('任务类型:', config['direction']['task_type'])
print('训练轮数:', config['training']['num_epochs'])
print('LoRA rank:', config['qlora']['r'])
print('最大序列长度:', config['training']['max_seq_length'])
print('Unsloth:', config['unsloth']['enabled'])
print('BF16:', config['training']['bf16'])
print('=' * 50)


---

## Stage 2: GPU检测

**作用**: 检测GPU型号/显存/BF16支持，给出推荐训练参数

**BF16 No**: RTX3060及以下，自动降级FP16

**显存不足警告**: 降低max_seq_length或batch_size


In [3]:
from src.gpu_detector import print_gpu_summary, GPUInfo

gpu_info = print_gpu_summary()
optimal = gpu_info.get_optimal_config()
print()
print('GPU 推荐训练参数:')
for k, v in optimal.items():
    print('  {}: {}'.format(k, v))

# 显存安全检查（填入你的模型参数量，单位：B）
MODEL_PARAMS_B = 8.0  # Qwen3-8B=8.0, Qwen3-4B=4.0
print()
print('显存安全检查: {}B 模型'.format(MODEL_PARAMS_B))
if gpu_info.check_memory_safe(MODEL_PARAMS_B):
    print('  [OK] 显存安全')
else:
    print('  [警告] 显存可能不足，建议降低max_seq_length或batch_size')



  GPU: NVIDIA GeForce RTX 4060 Laptop GPU
  VRAM: 8.0 GB
  CUDA: 12.6
  Architecture: Ada Lovelace (CC 8.9)
  BF16 Support: Yes


GPU 推荐训练参数:
  batch_size: 1
  gradient_accumulation_steps: 8
  max_seq_length: 1024
  lora_r: 8
  gradient_checkpointing: True
  bf16: True
  fp16: False

显存安全检查: 8.0B 模型
  [OK] 显存安全


---

## Stage 3: 数据加载

**支持格式(自动检测)**:
-Alpaca: instruction+output
-ShareGPT: conversations
-ChatML: messages
-CoT: Question+Complex_CoT+Response
-通用JSON: 含prompt/response/content

**放置数据**: ./datasets/*.jsonl 等

**可跳过**: 数据不变时不需要重复运行


In [ ]:
from src.data_loader import DatasetLoader, print_dataset_preview

print('数据集预览:')
try:
    print_dataset_preview(config['data']['dataset_dir'])
except Exception as e:
    print('预览失败:', e)
    print('请确认dataset_dir路径正确且存在数据文件')

print()
loader = DatasetLoader(config)
train_data, val_data, detected_fmt = loader.load()
print()
print('格式:', detected_fmt, '| 训练集:', len(train_data), '| 验证集:', len(val_data))
if train_data:
    print()
    print('第一个样本预览(前400字符):')
    print(train_data[0].get('text', '')[:400])


---

## Stage 4: 模型加载

**作用**: 加载基座+tokenizer，应用QLoRA 4bit量化

**显存占用(8B模型)**: 基座4GB+LoRA0.5GB+激活2-3GB=约8GB

**首次**: 会下载模型(约16GB)

**可跳过**: 模型不变时不需要重复运行


In [ ]:
from src.trainer import load_model

print('加载模型(首次可能需要几分钟)...')
model, tokenizer, use_unsloth = load_model(config, gpu_info.get_optimal_config())
print()
print('=' * 40)
print('模型加载完成!')
print('  Unsloth:', use_unsloth)
print('  设备:', next(model.parameters()).device)
print('=' * 40)


---

## Stage 5: 训练

**支持**: EarlyStopping(连续3次eval_loss无改善自动停)+WandB

**输出**: outputs/目录保存最终模型

**中断**: Ctrl+Enter可安全中断

**恢复**: config['training']['resume_from_checkpoint']='outputs/checkpoint-xxx'


In [ ]:
from src.trainer import train

print('=' * 40)
print('开始训练!')
print('=' * 40)
print('轮数:', config['training']['num_epochs'], '| LoRA r:', config['qlora']['r'], '| Unsloth:', config['unsloth']['enabled'])
print('Ctrl+Enter可安全中断')
print()

final_model_dir = train(config, gpu_info.get_optimal_config())

print()
print('=' * 40)
print('训练完成! 保存路径:', final_model_dir)
print('=' * 40)


---

## Stage 6: 评估+推理监控

**Part A**: loss曲线+BERTScore+Rouge

**Part B**: 样例推理+详细性能报告

### 性能指标

|指标|说明|评判|
|---|---|---|
|First Token Latency|首Token时间|<0.5s优秀|
|Pure Gen Speed|生成速度|>30优秀|
|GPU Memory Peak|显存峰值|<6GB优秀|
|GPU Utilization|GPU利用率|>85%优秀|

### 速度评级(tokens/s): >30优秀 / 20-30良好 / <10需优化


In [ ]:
from src.evaluator import evaluate_training, generate_sample_output
from transformers import AutoTokenizer
import torch

MODEL_PATH = 'outputs/final_model'

print('加载tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Part A: 训练评估
print()
print('=' * 50)
print('Part A: 训练评估(Loss曲线+Metrics)')
print('=' * 50)
eval_result = evaluate_training(config, MODEL_PATH)
print('报告: outputs/training_report.png')

# Part B: 推理+性能监控
print()
print('=' * 50)
print('Part B: 推理性能监控')
print('=' * 50)
sample_results = generate_sample_output(
    model_path=MODEL_PATH,
    tokenizer=tokenizer,
    num_samples=2,
    max_length=512,
    system_prompt=config['direction']['system_prompt'],
)

print()
print('评估完成! 下一步: Cell 7(自定义推理) 或 Cell 8(导出)')


---

## Stage 7: 独立推理测试(完全可自定义)

**使用场景**: 训练中测试/导出后验证/对比checkpoint

### 可调参数

|TEST_PROMPT|测试问题|改成你的|
|MAX_NEW_TOKENS|最大长度|256-1024|
|TEMPERATURE|随机性|0.1-0.9|

**运行后可看到**: 生成速度/GPU温度-功耗-利用率/显存峰值/速度评级


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from src.performance_monitor import AdvancedPerformanceMonitor, print_detailed_performance_report
from threading import Thread
import time, torch

# 【修改这里】
MODEL_PATH = 'outputs/final_model'
TEST_PROMPT = '请介绍一下人工智能的发展历史。'
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.7

print('加载模型:', MODEL_PATH)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, quantization_config=bnb, device_map='auto', trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

try:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
    print('Unsloth推理模式已启用')
except:
    print('标准HF推理模式')

# 准备输入
messages = [{'role':'system','content':config['direction']['system_prompt']},
           {'role':'user','content':TEST_PROMPT}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)
input_token_count = inputs.input_ids.shape[1]
print('输入Tokens:', input_token_count)
print('生成中...')

# 监控+流式推理
monitor = AdvancedPerformanceMonitor()
monitor.tokenizer = tokenizer
monitor.start()

streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

def bg():
    while monitor.monitoring:
        monitor.collect_snapshot()
        time.sleep(0.1)

Thread(target=bg, daemon=True).start()

kwargs = dict(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask,
              max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE,
              top_p=0.9, use_cache=True, streamer=streamer, do_sample=True)

monitor.start_inference_timing(input_token_count)
Thread(target=model.generate, kwargs=kwargs).start()

first = False
generated = ''
print('-' * 40)
print('回答:')
for t in streamer:
    if not first:
        monitor.record_first_token()
        first = True
    monitor.record_token_generation(t)
    print(t, end='', flush=True)
    generated += t

monitor.generation_end_time = time.time()
monitor.stop()
monitor.output_tokens = len(tokenizer.encode(generated))

print()
print('-' * 40)
print_detailed_performance_report(
    monitor.calculate_speeds(),
    monitor.get_summary(),
    monitor.get_memory_analysis()
)

del model; torch.cuda.empty_cache()


---

## Stage 8: 导出模型

**作用**: 导出为可部署格式

|HF safetensors|标准格式|from_pretrained()直接加载|
|GGUF Q4_K_M|Ollama量化|推荐，4.8GB/精度平衡|

**GGUF量化级别**: Q4_K_M(推荐)/Q5_K_S/Q8_0/F16


In [ ]:
from src.exporter import export_model

MODEL_TO_EXPORT = 'outputs/final_model'
config['export']['format'] = 'both'   # hf / gguf / both
config['export']['gguf_quant'] = 'Q4_K_M'

print('导出:', MODEL_TO_EXPORT)
results = export_model(config, MODEL_TO_EXPORT)
print('完成!')
for fmt, path in results.items():
    print(' ', fmt+':', path)
